In [16]:
#------------------------------------------------------------------------------------------------
# Question 4: GenAI Clinical Data Assistant (Mock LLM Implementation)
# Author: Adashi Odama
# Step 1: Load and Explore AE Dataset
#
# Using ADAE as specified in the assessment even though (pharmaversesdtm::ae) was also referenced
# because ADAE contains the same required variables (USUBJID, AESEV, AETERM, AESOC)
# in an analysis-ready format.
#Also included TRTEMFL and SAFFL as these are critical population flag used in AE analysis and reporting
#--------------------------------------------------------------------------------------------------


#--------------------------------------------------------------------------------------------------
# Step 1: Load AE Dataset with Explicit Path
#---------------------------------------------------------------------------------------------------

import pandas as pd

file_path = "/adae.csv"

ae = pd.read_csv(file_path)
print(f"✓ Loaded from: {file_path}")



✓ Loaded from: /adae.csv


In [20]:
#---------------------------------------------------------------------------------------------------
# Step 2: Explore ADAE Dataset Structure
#----------------------------------------------------------------------------------------------------

# Display dataset information
print("="*80)
print("ADAE Dataset Overview")
print("="*80)
print(f"Shape: {ae.shape[0]} rows x {ae.shape[1]} columns\n")

# Check for required columns
required_cols = ['USUBJID', 'AESEV', 'AETERM', 'AESOC', 'SAFFL', 'TRTEMFL']
print("Required Columns Check:")
for col in required_cols:
    if col in ae.columns:
        print(f"  ✓ {col}")
    else:
        print(f"  ✗ {col} - MISSING!")
print()

# Display sample data
print("Sample Data (first 5 rows):")
print(ae[required_cols].head())
print()

# Show unique values for key columns
print("Data Summary:")
print(f"  - Unique subjects: {ae['USUBJID'].nunique()}")
print(f"  - Unique aeterm: {ae['AETERM'].nunique()}")
print(f"  - Total AE records: {len(ae)}")
print(f"Safety Population flag:")
print(ae['SAFFL'].value_counts())
print(f"Treatment Emergent flag:")
print(ae['TRTEMFL'].value_counts())
print(f"\nSeverity levels (AESEV):")
print(ae['AESEV'].value_counts())
print(f"\nTop 10 AE Terms (AETERM):")
print(ae['AETERM'].value_counts().head(10))
print(f"\nSystem Organ Classes (AESOC):")
print(ae['AESOC'].value_counts())

ADAE Dataset Overview
Shape: 1191 rows x 107 columns

Required Columns Check:
  ✓ USUBJID
  ✓ AESEV
  ✓ AETERM
  ✓ AESOC
  ✓ SAFFL
  ✓ TRTEMFL

Sample Data (first 5 rows):
       USUBJID     AESEV                     AETERM  \
0  01-701-1015      MILD  APPLICATION SITE ERYTHEMA   
1  01-701-1015      MILD  APPLICATION SITE PRURITUS   
2  01-701-1015      MILD                  DIARRHOEA   
3  01-701-1023  MODERATE                   ERYTHEMA   
4  01-701-1023      MILD                   ERYTHEMA   

                                               AESOC SAFFL TRTEMFL  
0  GENERAL DISORDERS AND ADMINISTRATION SITE COND...     Y       Y  
1  GENERAL DISORDERS AND ADMINISTRATION SITE COND...     Y       Y  
2                         GASTROINTESTINAL DISORDERS     Y       Y  
3             SKIN AND SUBCUTANEOUS TISSUE DISORDERS     Y       Y  
4             SKIN AND SUBCUTANEOUS TISSUE DISORDERS     Y       Y  

Data Summary:
  - Unique subjects: 225
  - Unique aeterm: 242
  - Total AE records

In [24]:
#------------------------------------------------------------------------------------------------------------------------
# Step 3: Define Schema for Mock LLM
#
# This schema describes the ADAE dataset columns based on actual data analysis.
# The Mock LLM will use this to map natural language questions to the correct
# column and understand valid values.
#
# Data Summary:
# - 1,191 AE records from 225 unique subjects
# - 242 unique adverse event terms (AETERM)
# - 23 System Organ Classes (AESOC)
# - All records have SAFFL='Y' and most have TRTEMFL='Y' (treatment-emergent)
#------------------------------------------------------------------------------------------------------------------------

ADAE_SCHEMA = {
    "USUBJID": {
        "description": "Unique Subject Identifier",
        "type": "identifier",
        "example_values": ["01-701-1015", "01-701-1023", "01-701-1028"],
        "keywords": ["subject", "patient", "participant", "ID", "identifier", "USUBJID"]
    },

    "AETERM": {
        "description": "Reported Term for the Adverse Event",
        "type": "categorical",
        "example_values": ["PRURITUS", "APPLICATION SITE PRURITUS", "ERYTHEMA", "APPLICATION SITE IRRITATION"
                          "RASH", "DIZZINESS", "HEADACHE", "NAUSEA","APPLICATION SITE ERYTHEMA" "DIARRHOEA"],
        "most_common": ["PRURITUS (84)", "APPLICATION SITE PRURITUS (78)",
                       "ERYTHEMA (59)", "RASH (45)", "DIZZINESS (34)"],
        "keywords": ["condition", "event", "symptom", "adverse event", "AE term",
                     "diagnosis", "reported term", "specific condition", "AETERM",
                     "pruritus", "headache", "nausea", "rash", "dizziness" ,"irritation","diarrhoea"]
    },

    "AESEV": {
        "description": "Severity/Intensity of Adverse Event",
        "type": "categorical",
        "valid_values": ["MILD", "MODERATE", "SEVERE"],
        "value_counts": {"MILD": 770, "MODERATE": 378, "SEVERE": 43},
        "keywords": ["severity", "intensity", "grade", "how severe", "how serious",
                     "how intense" "mild", "moderate", "severe", "AESEV"]
    },

    "AESOC": {
        "description": "Primary System Organ Class (MedDRA classification)",
        "type": "categorical",
        "example_values": [
            "GENERAL DISORDERS AND ADMINISTRATION SITE CONDITIONS",
            "SKIN AND SUBCUTANEOUS TISSUE DISORDERS",
            "NERVOUS SYSTEM DISORDERS",
            "CARDIAC DISORDERS",
            "GASTROINTESTINAL DISORDERS",
            "INFECTIONS AND INFESTATIONS"
        ],
        "most_common": [
            "GENERAL DISORDERS (292)",
            "SKIN DISORDERS (276)",
            "NERVOUS SYSTEM (101)",
            "CARDIAC (91)"
        ],
        "keywords": ["body system", "organ class", "system", "organ", "SOC", "AESOC",
                     "cardiac", "heart", "nervous", "brain", "skin", "dermatology",
                     "gastrointestinal", "stomach", "gut", "digestive",
                     "respiratory", "lung", "breathing"]
    },

    "SAFFL": {
        "description": "Safety Population Flag (Y/N)",
        "type": "flag",
        "valid_values": ["Y", "N"],
        "note": "All records in this dataset have SAFFL='Y'",
        "keywords": ["safety population", "safety flag", "safety analysis set", "SAFFL"]
    },

     "TRTEMFL": {
        "description": "Treatment Emergent Analysis Flag (Y/blank)",
        "type": "flag",
        "valid_values": ["Y", ""],
        "note": "1,122 of 1,191 records have TRTEMFL='Y', rest are blank",
        "keywords": ["treatment emergent", "treatment-emergent", "TEAE",
                     "emergent", "TRTEMFL", "during treatment"]
    }
}

# Display schema summary
print("="*80)
print("ADAE Schema Defined for Mock LLM")
print("="*80)
print(f"\nColumns defined: {list(ADAE_SCHEMA.keys())}")
print(f"\nSample keyword mappings:")
for col in ["AETERM", "AESEV", "AESOC", "SAFFL","TRTEMFL"]:
    print(f"\n{col}:")
    print(f"  Keywords: {', '.join(ADAE_SCHEMA[col]['keywords'][:5])}...")
    if 'example_values' in ADAE_SCHEMA[col]:
        print(f"  Examples: {', '.join(ADAE_SCHEMA[col]['example_values'][:3])}...")

ADAE Schema Defined for Mock LLM

Columns defined: ['USUBJID', 'AETERM', 'AESEV', 'AESOC', 'SAFFL', 'TRTEMFL']

Sample keyword mappings:

AETERM:
  Keywords: condition, event, symptom, adverse event, AE term...
  Examples: PRURITUS, APPLICATION SITE PRURITUS, ERYTHEMA...

AESEV:
  Keywords: severity, intensity, grade, how severe, how serious...

AESOC:
  Keywords: body system, organ class, system, organ, SOC...
  Examples: GENERAL DISORDERS AND ADMINISTRATION SITE CONDITIONS, SKIN AND SUBCUTANEOUS TISSUE DISORDERS, NERVOUS SYSTEM DISORDERS...

SAFFL:
  Keywords: safety population, safety flag, safety analysis set, SAFFL...

TRTEMFL:
  Keywords: treatment emergent, treatment-emergent, TEAE, emergent, TRTEMFL...


In [36]:
#-------------------------------------------------------------------------------------------------------------
# Step 4: Create Mock LLM Class - ClinicalTrialDataAgent (IMPROVED)
#
# This class simulates an LLM by using rule-based keyword matching to parse
# natural language questions into structured queries.
#
# Logic Flow: Question → Parse → Structured Output (JSON)
#--------------------------------------------------------------------------------------------------------------

import json
import re

class ClinicalTrialDataAgent:
    """
    Mock LLM agent that translates natural language questions about adverse events
    into structured Pandas queries.

    Simulates LLM behavior using keyword matching against the ADAE schema.
    """

    def __init__(self, schema):
        """
        Initialize the agent with the ADAE schema.

        Args:
            schema (dict): ADAE_SCHEMA dictionary containing column descriptions
        """
        self.schema = schema

    def parse_question(self, question):
        """
        Parse a natural language question into structured JSON output.

        This mock implementation uses keyword matching to determine:
        1. Which column to filter (target_column)
        2. What value to search for (filter_value)

        Args:
            question (str): Natural language question from user

        Returns:
            dict: Structured JSON with keys:
                - target_column: The column to filter
                - filter_value: The value to search for
                - reasoning: Explanation of how the decision was made
        """
        # Convert question to lowercase for matching
        question_lower = question.lower()

        # Initialize result
        result = {
            "target_column": None,
            "filter_value": None,
            "reasoning": ""
        }

        # IMPROVED LOGIC: Check for specific values FIRST before generic keywords

        # Check for severity values (highest priority)
        if any(word in question_lower for word in ['mild', 'moderate', 'severe', 'severity', 'intensity']):
            result['target_column'] = 'AESEV'
            if 'mild' in question_lower:
                result['filter_value'] = 'MILD'
            elif 'moderate' in question_lower:
                result['filter_value'] = 'MODERATE'
            elif 'severe' in question_lower:
                result['filter_value'] = 'SEVERE'
            result['reasoning'] = f"Detected severity question → AESEV={result['filter_value']}"
            return result

        # Check for treatment-emergent flags
        if any(phrase in question_lower for phrase in ['treatment emergent', 'treatment-emergent', 'teae', 'emergent']):
            result['target_column'] = 'TRTEMFL'
            result['filter_value'] = 'Y'
            result['reasoning'] = "Detected treatment-emergent question → TRTEMFL='Y'"
            return result

        # Check for safety population flags
        if 'safety population' in question_lower or 'safety flag' in question_lower:
            result['target_column'] = 'SAFFL'
            result['filter_value'] = 'Y'
            result['reasoning'] = "Detected safety population question → SAFFL='Y'"
            return result

        # Check for System Organ Class keywords
        soc_mappings = {
            'cardiac': 'CARDIAC DISORDERS',
            'heart': 'CARDIAC DISORDERS',
            'skin': 'SKIN AND SUBCUTANEOUS TISSUE DISORDERS',
            'dermatology': 'SKIN AND SUBCUTANEOUS TISSUE DISORDERS',
            'dermatological': 'SKIN AND SUBCUTANEOUS TISSUE DISORDERS',
            'nervous': 'NERVOUS SYSTEM DISORDERS',
            'brain': 'NERVOUS SYSTEM DISORDERS',
            'neurological': 'NERVOUS SYSTEM DISORDERS',
            'gastrointestinal': 'GASTROINTESTINAL DISORDERS',
            'stomach': 'GASTROINTESTINAL DISORDERS',
            'digestive': 'GASTROINTESTINAL DISORDERS',
            'gut': 'GASTROINTESTINAL DISORDERS',
            'respiratory': 'RESPIRATORY, THORACIC AND MEDIASTINAL DISORDERS',
            'lung': 'RESPIRATORY, THORACIC AND MEDIASTINAL DISORDERS',
            'breathing': 'RESPIRATORY, THORACIC AND MEDIASTINAL DISORDERS'
        }

        for key, value in soc_mappings.items():
            if key in question_lower:
                result['target_column'] = 'AESOC'
                result['filter_value'] = value
                result['reasoning'] = f"Detected organ system '{key}' → AESOC='{value}'"
                return result

        # Check for specific AE terms (medical conditions)
        term_mappings = {
            'headache': 'HEADACHE',
            'nausea': 'NAUSEA',
            'dizziness': 'DIZZINESS',
            'rash': 'RASH',
            'itching': 'PRURITUS',
            'pruritus': 'PRURITUS',
            'diarrhea': 'DIARRHOEA',
            'diarrhoea': 'DIARRHOEA',
            'fatigue': 'FATIGUE',
            'cough': 'COUGH',
            'erythema': 'ERYTHEMA'
        }

        for key, value in term_mappings.items():
            if key in question_lower:
                result['target_column'] = 'AETERM'
                result['filter_value'] = value
                result['reasoning'] = f"Detected specific condition '{key}' → AETERM='{value}'"
                return result

        # Fallback: If no specific match, indicate unclear question
        result['reasoning'] = "Could not identify specific filter criteria"
        return result

    def format_output(self, parsed_result):
        """
        Format the parsed result as clean JSON output.

        Args:
            parsed_result (dict): Output from parse_question()

        Returns:
            str: Pretty-printed JSON string
        """
        return json.dumps(parsed_result, indent=2)


# Re-initialize the agent with our schema
agent = ClinicalTrialDataAgent(ADAE_SCHEMA)

print("="*80)
print("✓ ClinicalTrialDataAgent Created (IMPROVED)")
print("="*80)
print("\nThe agent now prioritizes:")
print("  1. Severity values (mild/moderate/severe)")
print("  2. Flags (treatment-emergent, safety)")
print("  3. Organ systems (cardiac, skin, nervous, etc.)")
print("  4. Specific AE terms (headache, nausea, etc.)")

✓ ClinicalTrialDataAgent Created (IMPROVED)

The agent now prioritizes:
  1. Severity values (mild/moderate/severe)
  2. Flags (treatment-emergent, safety)
  3. Organ systems (cardiac, skin, nervous, etc.)
  4. Specific AE terms (headache, nausea, etc.)


In [37]:
#-------------------------------------------------------------------------------------------------------
# Step 5: Test Mock LLM with Three Example Queries
#
# As specified in the assessment deliverables.
#-------------------------------------------------------------------------------------------------------

print("="*80)
print("Testing ClinicalTrialDataAgent - Example Queries")
print("="*80)

# Three diverse, clinically relevant test questions
test_questions = [
    # Query 1: Severity-based (regulatory focus)
    "Identify all subjects who experienced severe adverse events",

    # Query 2: System-based (medical review)
    "Which subjects had nervous system disorders?",

    # Query 3: Specific condition (signal detection)
    "Show me all patients who reported pruritus"
]

print("\n Running Test Queries...\n")

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*80}")
    print(f"Query {i}: {question}")
    print('='*80)

    # Parse the question
    result = agent.parse_question(question)

    # Display the structured output
    print(f"\nStructured Output:")
    print(agent.format_output(result))

    # Show what would be queried
    if result['target_column'] and result['filter_value']:
        print(f"\n✓ Filter: ae[ae['{result['target_column']}'] == '{result['filter_value']}']")
        print(f"   Reasoning: {result['reasoning']}")
    else:
        print(f"\n Could not parse question")

print("\n" + "="*80)
print("✓ Testing Complete")
print("="*80)

Testing ClinicalTrialDataAgent - Example Queries

 Running Test Queries...


Query 1: Identify all subjects who experienced severe adverse events

Structured Output:
{
  "target_column": "AESEV",
  "filter_value": "SEVERE",
  "reasoning": "Detected severity question \u2192 AESEV=SEVERE"
}

✓ Filter: ae[ae['AESEV'] == 'SEVERE']
   Reasoning: Detected severity question → AESEV=SEVERE

Query 2: Which subjects had nervous system disorders?

Structured Output:
{
  "target_column": "AESOC",
  "filter_value": "NERVOUS SYSTEM DISORDERS",
  "reasoning": "Detected organ system 'nervous' \u2192 AESOC='NERVOUS SYSTEM DISORDERS'"
}

✓ Filter: ae[ae['AESOC'] == 'NERVOUS SYSTEM DISORDERS']
   Reasoning: Detected organ system 'nervous' → AESOC='NERVOUS SYSTEM DISORDERS'

Query 3: Show me all patients who reported pruritus

Structured Output:
{
  "target_column": "AETERM",
  "filter_value": "PRURITUS",
  "reasoning": "Detected specific condition 'pruritus' \u2192 AETERM='PRURITUS'"
}

✓ Filter: ae[ae['

In [38]:
#--------------------------------------------------------------------------------------------------------
# Step 6: Query Execution Function
#
# This function takes the LLM's parsed output and applies the actual Pandas
# filter to the ADAE dataframe, then returns results.
#--------------------------------------------------------------------------------------------------------

def query_exec(parsed_result, dataframe):
    """
    Execute a query on the ADAE dataframe based on parsed LLM output.

    Args:
        parsed_result (dict): Output from agent.parse_question()
                             Contains 'target_column' and 'filter_value'
        dataframe (pd.DataFrame): The ADAE dataframe to query

    Returns:
        dict: Results containing:
            - subject_count: Number of unique subjects matching the filter
            - subject_ids: List of unique USUBJIDs
            - record_count: Total number of AE records matching
            - query_summary: Human-readable summary
    """
    # Initialize result
    result = {
        "subject_count": 0,
        "subject_ids": [],
        "record_count": 0,
        "query_summary": ""
    }

    # Check if parsing was successful
    if not parsed_result['target_column'] or not parsed_result['filter_value']:
        result['query_summary'] = "Could not execute - no valid filter detected"
        return result

    # Extract filter parameters
    target_col = parsed_result['target_column']
    filter_val = parsed_result['filter_value']

    # Apply the Pandas filter
    try:
        filtered_df = dataframe[dataframe[target_col] == filter_val]

        # Get unique subjects
        unique_subjects = filtered_df['USUBJID'].unique().tolist()

        # Populate results
        result['subject_count'] = len(unique_subjects)
        result['subject_ids'] = sorted(unique_subjects)
        result['record_count'] = len(filtered_df)
        result['query_summary'] = (
            f"Found {result['subject_count']} unique subject(s) "
            f"with {result['record_count']} matching AE record(s)"
        )

    except KeyError as e:
        result['query_summary'] = f"ERROR: Column '{target_col}' not found in dataframe"
    except Exception as e:
        result['query_summary'] = f"ERROR executing query: {str(e)}"

    return result


# Test the execution function
print("="*80)
print("✓ Query Execution Function Created")
print("="*80)
print("\nFunction: query_exec(parsed_result, dataframe)")
print("\nReturns:")
print("  - subject_count: Number of unique subjects")
print("  - subject_ids: List of USUBJIDs")
print("  - record_count: Total AE records")
print("  - query_summary: Human-readable result")

✓ Query Execution Function Created

Function: query_exec(parsed_result, dataframe)

Returns:
  - subject_count: Number of unique subjects
  - subject_ids: List of USUBJIDs
  - record_count: Total AE records
  - query_summary: Human-readable result


In [39]:

#-----------------------------------------------------------------------------------------------------
# Step 7: Complete End-to-End Test
#
# Demonstrates the full workflow: Question → Parse → Execute → Results
# Returns COUNT of unique subjects and FULL LIST of matching IDs
#-----------------------------------------------------------------------------------------------------

print("\n" + "="*80)
print("COMPLETE END-TO-END TEST: Question → Parse → Execute → Results")
print("="*80)

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*80}")
    print(f"TEST QUERY {i}")
    print('='*80)
    print(f"Question: {question}")
    print()

    # Step 1: Parse the question
    parsed = agent.parse_question(question)
    print(f"📋 Parsed Output:")
    print(f"   Target Column: {parsed['target_column']}")
    print(f"   Filter Value: {parsed['filter_value']}")
    print(f"   Reasoning: {parsed['reasoning']}")
    print()

    # Step 2: Execute the query
    results = query_exec(parsed, ae)
    print(f" Query Results:")
    print(f"   {results['query_summary']}")

    if results['subject_count'] > 0:
        print(f"\n  Complete List of Unique Subject IDs ({results['subject_count']} subjects):")
        for subj_id in results['subject_ids']:
            print(f"      - {subj_id}")
    else:
        print(f"\n  No subjects found matching this criteria")

    print()

print("="*80)
print("✓ END-TO-END TESTING COMPLETE")
print("="*80)
print("\n Clinical Trial Data Agent Successfully Demonstrated!")
print("\nWorkflow:")
print("  1. ✓ Natural language question received")
print("  2. ✓ Mock LLM parses into structured JSON")
print("  3. ✓ Pandas filter applied to ADAE dataframe")
print("  4. ✓ Results returned with COUNT and FULL LIST of subject IDs")
print("\n Deliverables Met:")
print("  ✓ ClinicalTrialDataAgent class implemented")
print("  ✓ Three example queries tested")
print("  ✓ Returns count of unique subjects (USUBJID)")
print("  ✓ Returns complete list of matching subject IDs")


COMPLETE END-TO-END TEST: Question → Parse → Execute → Results

TEST QUERY 1
Question: Identify all subjects who experienced severe adverse events

📋 Parsed Output:
   Target Column: AESEV
   Filter Value: SEVERE
   Reasoning: Detected severity question → AESEV=SEVERE

 Query Results:
   Found 31 unique subject(s) with 43 matching AE record(s)

  Complete List of Unique Subject IDs (31 subjects):
      - 01-701-1211
      - 01-703-1086
      - 01-703-1119
      - 01-703-1175
      - 01-704-1008
      - 01-704-1135
      - 01-704-1445
      - 01-705-1393
      - 01-706-1049
      - 01-708-1019
      - 01-708-1178
      - 01-708-1272
      - 01-708-1428
      - 01-709-1007
      - 01-709-1259
      - 01-710-1070
      - 01-710-1077
      - 01-710-1083
      - 01-710-1154
      - 01-710-1271
      - 01-710-1368
      - 01-711-1143
      - 01-714-1195
      - 01-716-1103
      - 01-716-1189
      - 01-717-1174
      - 01-718-1066
      - 01-718-1079
      - 01-718-1170
      - 01-718-1371

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
